### Pydantic 모델
- 정의된 데이터만 전송되도록 요청 바디 검증

In [ ]:
from datetime import datetime

from pydantic import BaseModel, PositiveInt,ValidationError

# 1. Pydantic 모델 생성
class User(BaseModel):
    id: int  
    name: str = 'John Doe'  
    signup_ts: datetime | None  
    tastes: dict[str, PositiveInt]  

# 2. 객체 생성
external_data = {
    'id': 123,
    'signup_ts': '2019-06-01 12:22',  
    'tastes': {
        'wine': 9,
        b'cheese': 7,  
        'cabbage': '1',  
    },
}

# user = User(id=1234,signup_ts=) 
# 이러한 형태로 객체로 만듬
user = User(**external_data)  # ** 은 이러한 형태로 객체 만들어줘 라는 의미

print(user.id)  
#> 123
print(user.model_dump())  # 모델의 필드와 값을 dict 형태로 반환


123
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [ ]:
# 3. 잘 만들어 졌는데 에러 확인
external_data = {'id': 'not an int', 'tastes': {}}  
try:
    User(**external_data)  
except ValidationError as e:
    print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [12]:
# Fruit : namd:string , color:"red" or "green", weight: float, bazam-dict:{key:str,tuple[int,bool,float]}

from typing import Annotated,Literal
from annotated_types import Gt

class Fruit(BaseModel):
    namd:str
    color:Literal["red","green"]
    weight:Annotated[float,Gt(0)]
    bazam:dict[str,list[tuple[int,bool,float]]]

In [ ]:
fruit = Fruit(name="apple",color="red",weight=4.2,bazam={'foobar':[(1,True,0.1)]})


In [ ]:
fruit = Fruit(name="apple",color="red",weight=4.2)
fruit

In [17]:
# Book - code(4자리), category(소설,과학,역사,경제,자기계발,컴퓨터), title-문자, writer-문자, price-int(0보다 커야함)

from pydantic import StringConstraints

class Book(BaseModel):
    code:Annotated[str,StringConstraints(min_length=4,max_length=4)]
    category:Literal["소설","과학","역사","경제","자기계발","컴퓨터"]
    title:str
    writer:str
    price:Annotated[int,Gt(0)]

In [18]:
try:
    Book(code='1001',category='경제',price=25000)
except ValidationError as e:
    print(e.errors())

[{'type': 'missing', 'loc': ('title',), 'msg': 'Field required', 'input': {'code': '1001', 'category': '경제', 'price': 25000}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('writer',), 'msg': 'Field required', 'input': {'code': '1001', 'category': '경제', 'price': 25000}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [19]:
# User-id:int, name:문자, age 14~100까지, email, role='user'or'admin'
# ge,le => Gt 처럼 선언 가능

from pydantic import Field

class User(BaseModel):
    id:int
    name:str
    age:Annotated[int,Field(ge=14,le=100)]
    email:str
    role:Literal["user","admin"]

In [20]:
try:
    User(id=1,name='홍길동',age=101,email="abc",role='user')
except ValidationError as e:
    print(e.errors())

[{'type': 'less_than_equal', 'loc': ('age',), 'msg': 'Input should be less than or equal to 100', 'input': 101, 'ctx': {'le': 100}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}]


In [ ]:
# Product => code:str, name:str, category:['food','clothes','electronics'], price:int ,0 >
# stock:int,0 >=

class Product(BaseModel):
    code:str
    name:str
    category:Literal['food','clothes','electronics']
    price:Annotated[int,Gt(0)]
    stock:Annotated[int,Field(ge=0)]

In [29]:
from annotated_types import Ge,Le

try:
    Product(code="1001",name='옷',category="clothes",price=25000,stock=1)
except ValidationError as e:
    print(e.errors())

## 중첩 모델

In [ ]:
class Item(BaseModel):
    item:str
    status:str

class Cart(BaseModel):
    id:int
    item:Item
    

In [34]:
try:
    cart = Cart(id=1,item=Item(item='바지',status='판매'))

except ValidationError as e:
    print(e.errors())

In [35]:
class OrderItem(BaseModel):
    product_name:str
    quantity:Annotated[int,Gt(0)]
    price:Annotated[int,Gt(0)]

class Order(BaseModel):
    order_id:int
    customer:str
    items:list[OrderItem]

In [ ]:

try:
    order = Order(order_id=1001,customer="홍길동",items=[
        # key:value 형태로 넣음
        {"product_name":"노트북","quantity":1,"price":55000},
        {"product_name":"마우스","quantity":2,"price":80000}
    ])
except ValidationError as e:
    print(e.errors())

In [44]:
class Comment(BaseModel):
    writer:str
    content:str

class Post(BaseModel):
    id:int
    title:str
    writer:str
    content:str
    comments:list[Comment]

In [47]:
try:
    post = Post(id=1,title="제목",writer="홍길동",content="내용",comments=[
        {"writer":"홍길동","content":"내용1"},
        {"writer":"홍길동","content":"내용2"},
        {"writer":"홍길동","content":"내용3"}
    ])
except ValidationError as e:
    print(e.errors())